# DX 704 Week 7 Project

This week's project will investigate issues in a quadcopter controller based using a linear quadratic regulator.
You will start with an existing model of the system and logs from a quadcopter based on it, investigate discrepancies, and ultimately train a new model of the system dynamics.

The full project description and a template notebook are available on GitHub: [Project 7 Materials](https://github.com/bu-cds-dx704/dx704-project-07).


## Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Introduction

You've just joined a drone startup.
On your first day, you join your new team to watch a test flight for a new quadcopter prototype.
Watching the prototype fly, the team comments that it is not as smooth as usual and suspects that something is off in the controller.
They provide you a copy of the dynamics model and log data from the prototype to investigate.

The quadcopter control model is a slightly more sophisticated version of the 1D quadcopter problem previously considered.

The state vector $\mathbf{x}_t$ now includes an acceleration component, and the action vector now has a servo control for the throttle instead of a raw force component.
\begin{array}{rcl}
\mathbf{x}_t & = & \begin{bmatrix} x_t \\ v_t \\ a_t \end{bmatrix} \\
\mathbf{u_t} & = & \begin{bmatrix} u_t \end{bmatrix}
\end{array}

## Part 1: Reconstruct the Control Matrix

You are provided the dynamics model in the files `model-A.tsv`, `model-B.tsv`, `cost-Q.tsv` and `cost-R.tsv`.
Recompute the control matrix $\mathbf{K}$ to be used in the infinite horizon linear quadratic regulator problem.

In [19]:
# YOUR CHANGES HERE

# Imports
import numpy as np
import pandas as pd
from scipy.linalg import solve_discrete_are

In [20]:
# Load model matrices
A = pd.read_csv('model-A.tsv', sep='\t', index_col=0).values
B = pd.read_csv('model-B.tsv', sep='\t', index_col=0).values
Q = pd.read_csv('cost-Q.tsv', sep='\t', index_col=0).values
R = pd.read_csv('cost-R.tsv', sep='\t', index_col=0).values

In [21]:
# Display the loaded matrices to verify
print("Matrix A (dynamics):")
print(A)
print("\nMatrix B (control input):")
print(B)
print("\nMatrix Q (state cost):")
print(Q)
print("\nMatrix R (control cost):")
print(R)

Matrix A (dynamics):
[[1 1 0]
 [0 1 1]
 [0 0 1]]

Matrix B (control input):
[[0]
 [0]
 [1]]

Matrix Q (state cost):
[[5 0 0]
 [0 1 0]
 [0 0 2]]

Matrix R (control cost):
[[5]]


In [22]:
# Solve the discrete-time algebraic Riccati equation
P = solve_discrete_are(A, B, Q, R)

# Compute the control matrix K - For discrete-time LQR: K = (R + B^T P B)^(-1) B^T P A
K = np.linalg.inv(R + B.T @ P @ B) @ B.T @ P @ A

# Display the control matrix
print("Control matrix K:")
print(K)

Control matrix K:
[[0.33445985 1.30445413 1.85813088]]


Save $\mathbf{K}$ in a file "control-K-intended.tsv" with columns x, v and a.

In [23]:
# YOUR CHANGES HERE
# Save K to TSV file with columns x, v, a
K_df = pd.DataFrame(K, columns=['x', 'v', 'a'])
K_df.to_csv('control-K-intended.tsv', sep='\t', index=False)

# verify the saved file
print("\nSaved to 'control-K-intended.tsv'")
print("\nVerification:")
K_verify = pd.read_csv('control-K-intended.tsv', sep='\t')
print(K_verify)


Saved to 'control-K-intended.tsv'

Verification:
         x         v         a
0  0.33446  1.304454  1.858131


Submit "control-K-intended.tsv" in Gradescope.

## Part 2: Recompute the Actions for the Logged States

You get access to the log data for the prototype as the file "qc-log.tsv".
It conveniently saves all the state and actions made.
Recompute the actions based on your $\mathbf{K}$ matrix computed in part 1.

In [24]:
# YOUR CHANGES HERE
qc_log = pd.read_csv('qc-log.tsv', sep='\t')

# Display the log data to understand its structure
print("Log data shape:", qc_log.shape)
print("\nFirst few rows of log data:")
print(qc_log.head(10))
print("\nColumn names:")
print(qc_log.columns.tolist())
print("\nLog data info:")
print(qc_log.info())

# Load the control matrix K that we computed in Part 1
K = pd.read_csv('control-K-intended.tsv', sep='\t').values

print("\nControl matrix K:")
print(K)

Log data shape: (100, 5)

First few rows of log data:
   index         x         v         a         u
0      0 -5.000000  0.000000  0.000000  1.702188
1      1 -5.000000 -0.017022  1.531969 -1.263200
2      2 -5.018724  1.452683  0.548285 -1.249321
3      3 -3.420773  1.840779 -0.521275 -0.212127
4      4 -1.395916  1.163611 -0.764317  0.452895
5      5 -0.115944  0.316620 -0.433143  0.472507
6      6  0.232338 -0.131253 -0.051201  0.191396
7      7  0.087960 -0.168682  0.115935 -0.036724
8      8 -0.097591 -0.041309  0.094477 -0.097311
9      9 -0.143030  0.053548  0.016345 -0.052800

Column names:
['index', 'x', 'v', 'a', 'u']

Log data info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   index   100 non-null    int64  
 1   x       100 non-null    float64
 2   v       100 non-null    float64
 3   a       100 non-null    float64
 4   u       100 non-

In [25]:
# Extract state variables from the log
# Assuming columns are named x, v, a (or similar)
x_states = qc_log[['x', 'v', 'a']].values

print("\nFirst few state vectors:")
print(x_states[:5])

# Recompute the control actions using u = -K * x
# K is (1, 3) and x is (n, 3), so we need u = -x @ K.T to get (n, 1)
u_check = -x_states @ K.T

print("\nShape of computed actions:", u_check.shape)
print("\nFirst few computed actions:")
print(u_check[:10])


First few state vectors:
[[-5.          0.          0.        ]
 [-5.         -0.01702188  1.53196898]
 [-5.01872407  1.45268284  0.54828547]
 [-3.42077294  1.84077896 -0.52127491]
 [-1.39591608  1.16361118 -0.76431704]]

Shape of computed actions: (100, 1)

First few computed actions:
[[ 1.67229926]
 [-1.15209535]
 [-1.23518258]
 [-0.2885035 ]
 [ 0.36920157]
 [ 0.43059849]
 [ 0.18864361]
 [-0.02480317]
 [-0.08902456]
 [-0.05238364]]


Save your computed actions as "qc-check.tsv" with columns "index" and "u_check".

In [26]:
# YOUR CHANGES HERE
# Create the output dataframe
qc_check = pd.DataFrame({
    'index': qc_log.index,
    'u_check': u_check.flatten()
})

print("\nOutput dataframe:")
print(qc_check.head(10))

# Save to TSV file
qc_check.to_csv('qc-check.tsv', sep='\t', index=False)

print("\nSaved to 'qc-check.tsv'")

# Verify saved file
qc_check_verify = pd.read_csv('qc-check.tsv', sep='\t')
print("\nVerification:")
print(qc_check_verify.head(10))
print("\nSummary statistics:")
print(qc_check_verify.describe())


Output dataframe:
   index   u_check
0      0  1.672299
1      1 -1.152095
2      2 -1.235183
3      3 -0.288504
4      4  0.369202
5      5  0.430598
6      6  0.188644
7      7 -0.024803
8      8 -0.089025
9      9 -0.052384

Saved to 'qc-check.tsv'

Verification:
   index   u_check
0      0  1.672299
1      1 -1.152095
2      2 -1.235183
3      3 -0.288504
4      4  0.369202
5      5  0.430598
6      6  0.188644
7      7 -0.024803
8      8 -0.089025
9      9 -0.052384

Summary statistics:
            index       u_check
count  100.000000  1.000000e+02
mean    49.500000 -1.480382e-03
std     29.011492  2.482665e-01
min      0.000000 -1.235183e+00
25%     24.750000 -7.376580e-10
50%     49.500000  6.004615e-20
75%     74.250000  3.057214e-10
max     99.000000  1.672299e+00


Submit "qc-check.tsv" in Gradescope.

## Part 3: Reverse Engineer the Actual Control Matrix

Now that you have found a systematic difference between your computed actions and the logged actions, estimate $
$, the control matrix that was actually used to choose actions in the prototype.

Hint: With a linear quadratic regulator, the optimal actions are always linear combinations of the state that are calculated using the control matrix.
You can use linear regression to reverse-engineer the coefficients in the control matrix.

In [27]:
# YOUR CHANGES HERE

from sklearn.linear_model import LinearRegression

# Load the data
qc_log = pd.read_csv('qc-log.tsv', sep='\t')

# Verify
print("Log data loaded:")
print(qc_log.head())

# Prepare the data for linear regression
# X: state vectors (x, v, a)
# y: actual control actions (u)
X = qc_log[['x', 'v', 'a']].values
y = qc_log['u'].values

print("\nState vectors (X) shape:", X.shape)
print("Control actions (y) shape:", y.shape)
print("\nFirst few state vectors:")
print(X[:5])
print("\nFirst few actual control actions:")
print(y[:5])

Log data loaded:
   index         x         v         a         u
0      0 -5.000000  0.000000  0.000000  1.702188
1      1 -5.000000 -0.017022  1.531969 -1.263200
2      2 -5.018724  1.452683  0.548285 -1.249321
3      3 -3.420773  1.840779 -0.521275 -0.212127
4      4 -1.395916  1.163611 -0.764317  0.452895

State vectors (X) shape: (100, 3)
Control actions (y) shape: (100,)

First few state vectors:
[[-5.          0.          0.        ]
 [-5.         -0.01702188  1.53196898]
 [-5.01872407  1.45268284  0.54828547]
 [-3.42077294  1.84077896 -0.52127491]
 [-1.39591608  1.16361118 -0.76431704]]

First few actual control actions:
[ 1.70218775 -1.26320045 -1.24932102 -0.21212738  0.45289543]


In [28]:
# Fit linear regression model
# The control law is: u = -K * x
# So: u = -k_x * x - k_v * v - k_a * a
# Which is: u = beta_0 * x + beta_1 * v + beta_2 * a (with no intercept)
model = LinearRegression(fit_intercept=False)
model.fit(X, y)

print("\nLinear regression coefficients:")
print(model.coef_)

# Cell 5: Extract K_actual
# Since u = -K * x, and we fitted u = beta * x
# We have: -K = beta, so K = -beta
K_actual = -model.coef_.reshape(1, -1)

print("\nActual control matrix K_actual:")
print(K_actual)



Linear regression coefficients:
[-0.34043755 -1.30012023 -1.95011696]

Actual control matrix K_actual:
[[0.34043755 1.30012023 1.95011696]]


In [29]:
# Compare with intended control matrix
K_intended = pd.read_csv('control-K-intended.tsv', sep='\t').values

print("\nComparison:")
print("K_intended: ", K_intended)
print("K_actual:   ", K_actual)
print("\nDifference (K_actual - K_intended):")
print(K_actual - K_intended)

# Verify the fit quality
y_pred = model.predict(X)
r_squared = model.score(X, y)

print(f"\nR-squared score: {r_squared:.6f}")
print("\nFirst 10 actual vs predicted actions:")
comparison = pd.DataFrame({
    'actual_u': y[:10],
    'predicted_u': y_pred[:10],
    'difference': (y - y_pred)[:10]
})
print(comparison)


Comparison:
K_intended:  [[0.33445985 1.30445413 1.85813088]]
K_actual:    [[0.34043755 1.30012023 1.95011696]]

Difference (K_actual - K_intended):
[[ 0.0059777  -0.0043339   0.09198608]]

R-squared score: 1.000000

First 10 actual vs predicted actions:
   actual_u  predicted_u    difference
0  1.702188     1.702188  2.220446e-16
1 -1.263200    -1.263200  0.000000e+00
2 -1.249321    -1.249321  4.440892e-16
3 -0.212127    -0.212127  6.938894e-16
4  0.452895     0.452895  6.661338e-16
5  0.472507     0.472507  5.551115e-17
6  0.191396     0.191396  1.110223e-16
7 -0.036724    -0.036724 -2.775558e-17
8 -0.097311    -0.097311  1.387779e-17
9 -0.052800    -0.052800 -3.469447e-17


Save $\mathbf{K}_{\mathrm{actual}}$ in "control-K-actual.tsv" with the same format as "control-K-intended.tsv".

In [30]:
# YOUR CHANGES HERE

# Save K_actual to TSV file
K_actual_df = pd.DataFrame(K_actual, columns=['x', 'v', 'a'])
K_actual_df.to_csv('control-K-actual.tsv', sep='\t', index=False)

print("\nSaved to 'control-K-actual.tsv'")

# Verify saved file
K_actual_verify = pd.read_csv('control-K-actual.tsv', sep='\t')
print("\nVerification:")
print(K_actual_verify)


Saved to 'control-K-actual.tsv'

Verification:
          x        v         a
0  0.340438  1.30012  1.950117


Submit "control-k-actual.tsv" in Gradescope.

## Part 4: Recompute the System Dynamics from the Log Data

On further investigation, it turns out that the control matrix $\mathbf{K}$ in the prototype was modified to compensate for state prediction errors.
You would like to recompute the $\mathbf{A}$ and $\mathbf{B}$ matrices using the log data since they are used to predict the next states.
However, since the action vector $\mathbf{u}_t$ is linearly dependent on the state via $\mathbf{u}_t=-\mathbf{K} \mathbf{x}_t$, you need a new data set so you can separate the effects of the $\mathbf{A}$ and $\mathbf{B}$ matrices.
Your co-workers kindly provide a new file "qc-train.tsv" where noise was added to each action.
Estimate the true $\mathbf{A}$ and $\mathbf{B}$ matrices based on this file.

In [31]:
# YOUR CHANGES HERE

# Load training data
qc_train = pd.read_csv('qc-train.tsv', sep='\t')

print("Training data shape:", qc_train.shape)
print("\nFirst few rows:")
print(qc_train.head(10))
print("\nColumn names:")
print(qc_train.columns.tolist())

Training data shape: (100, 5)

First few rows:
   index         x         v         a         u
0      0 -5.000000  0.000000  0.000000  1.729856
1      1 -5.000000 -0.017299  1.556871 -1.311911
2      2 -5.019028  1.476577  0.531837 -1.198850
3      3 -3.394793  1.846154 -0.493944 -0.297565
4      4 -1.364024  1.195267 -0.811147  0.472619
5      5 -0.049230  0.300425 -0.466905  0.461118
6      6  0.281237 -0.177789 -0.098590  0.532618
7      7  0.085670 -0.258996  0.370908 -0.490600
8      8 -0.199226  0.124172 -0.033542  0.213700
9      9 -0.062636  0.077754  0.155434 -0.311302

Column names:
['index', 'x', 'v', 'a', 'u']


In [ ]:
# Understand the data structure
# We need to predict x_{t+1} from x_t and u_t
# The dynamics equation is: x_{t+1} = A * x_t + B * u_t

# Let's check if we have sequential states
print("\nChecking data structure:")
print(qc_train.describe())



Checking data structure:
            index           x           v           a           u
count  100.000000  100.000000  100.000000  100.000000  100.000000
mean    49.500000   -0.168632    0.046746    0.005247   -0.001217
std     29.011492    0.942702    0.286777    0.236616    0.350461
min      0.000000   -5.019028   -0.258996   -0.811147   -1.311911
25%     24.750000   -0.065679   -0.068826   -0.109545   -0.164375
50%     49.500000    0.002231    0.013166   -0.012370    0.015367
75%     74.250000    0.122542    0.086858    0.087861    0.140310
max     99.000000    0.371780    1.846154    1.556871    1.729856


In [33]:
# Prepare the data for regression
# We need to separate current states and next states
# Assuming the data has columns for current state and possibly next state
# or we need to shift the data

if 'x_next' in qc_train.columns:
    # If next states are explicitly provided
    X_current = qc_train[['x', 'v', 'a']].values
    u_current = qc_train[['u']].values
    X_next = qc_train[['x_next', 'v_next', 'a_next']].values
else:
    # If we need to infer next states from sequential data
    # x_{t+1} is the state at the next timestep
    X_current = qc_train[['x', 'v', 'a']].values[:-1]  # All but last
    u_current = qc_train[['u']].values[:-1]  # All but last
    X_next = qc_train[['x', 'v', 'a']].values[1:]  # All but first

print("\nCurrent states shape:", X_current.shape)
print("Current actions shape:", u_current.shape)
print("Next states shape:", X_next.shape)

print("\nFirst few samples:")
print("X_current:\n", X_current[:3])
print("u_current:\n", u_current[:3])
print("X_next:\n", X_next[:3])


Current states shape: (99, 3)
Current actions shape: (99, 1)
Next states shape: (99, 3)

First few samples:
X_current:
 [[-5.          0.          0.        ]
 [-5.         -0.01729856  1.55687058]
 [-5.01902842  1.47657746  0.53183744]]
u_current:
 [[ 1.7298562 ]
 [-1.31191133]
 [-1.19885   ]]
X_next:
 [[-5.         -0.01729856  1.55687058]
 [-5.01902842  1.47657746  0.53183744]
 [-3.39479321  1.84615378 -0.49394382]]


In [34]:
# Set up the regression problem
# For each component of x_{t+1}, we fit:
# x_{t+1} = A * x_t + B * u_t
# This means each row of x_{t+1} is predicted by [x_t, u_t]
# We need to stack x_t and u_t horizontally

# Combined input: [x, v, a, u]
X_combined = np.hstack([X_current, u_current])

print("\nCombined input shape:", X_combined.shape)
print("First few combined inputs:\n", X_combined[:3])



Combined input shape: (99, 4)
First few combined inputs:
 [[-5.          0.          0.          1.7298562 ]
 [-5.         -0.01729856  1.55687058 -1.31191133]
 [-5.01902842  1.47657746  0.53183744 -1.19885   ]]


In [36]:
# Fit the separate regression for each state component
# x_next, v_next, a_next are the three outputs

models = []
A_rows = []
B_rows = []

for i, state_name in enumerate(['x', 'v', 'a']):
    # Fit regression for each state component
    model = LinearRegression(fit_intercept=False)
    model.fit(X_combined, X_next[:, i])
    models.append(model)
    
    # Extract coefficients
    # First 3 coefficients are A matrix row
    # Last coefficient is B matrix row
    A_row = model.coef_[:3]
    B_row = model.coef_[3:]
    
    A_rows.append(A_row)
    B_rows.append(B_row)
    
    print(f"\n{state_name}_next coefficients:")
    print(f"  A row: {A_row}")
    print(f"  B row: {B_row}")
    print(f"  R² score: {model.score(X_combined, X_next[:, i]):.6f}")



x_next coefficients:
  A row: [1.00000000e+00 1.10000000e+00 9.99200722e-16]
  B row: [-5.55111512e-17]
  R² score: 1.000000

v_next coefficients:
  A row: [5.8201259e-17 9.0000000e-01 9.5000000e-01]
  B row: [-0.01]
  R² score: 1.000000

a_next coefficients:
  A row: [-9.31378031e-17  5.55111512e-16  1.10000000e+00]
  B row: [0.9]
  R² score: 1.000000


In [38]:
# Construct A and B matrices
A_new = np.array(A_rows)
B_new = np.array(B_rows)

print("Estimated system dynamics")
print("\nMatrix A (new):")
print(A_new)
print("\nMatrix B (new):")
print(B_new)

# Compare with original matrices
A_original = pd.read_csv('model-A.tsv', sep='\t', index_col=0).values
B_original = pd.read_csv('model-B.tsv', sep='\t', index_col=0).values

print("Comparison with the original matrices")
print("\nMatrix A (original):")
print(A_original)
print("\nMatrix A (new):")
print(A_new)
print("\nDifference A (new - original):")
print(A_new - A_original)

print("\nMatrix B (original):")
print(B_original)
print("\nMatrix B (new):")
print(B_new)
print("\nDifference B (new - original):")
print(B_new - B_original)


Estimated system dynamics

Matrix A (new):
[[ 1.00000000e+00  1.10000000e+00  9.99200722e-16]
 [ 5.82012590e-17  9.00000000e-01  9.50000000e-01]
 [-9.31378031e-17  5.55111512e-16  1.10000000e+00]]

Matrix B (new):
[[-5.55111512e-17]
 [-1.00000000e-02]
 [ 9.00000000e-01]]
Comparison with the original matrices

Matrix A (original):
[[1 1 0]
 [0 1 1]
 [0 0 1]]

Matrix A (new):
[[ 1.00000000e+00  1.10000000e+00  9.99200722e-16]
 [ 5.82012590e-17  9.00000000e-01  9.50000000e-01]
 [-9.31378031e-17  5.55111512e-16  1.10000000e+00]]

Difference A (new - original):
[[ 8.88178420e-16  1.00000000e-01  9.99200722e-16]
 [ 5.82012590e-17 -1.00000000e-01 -5.00000000e-02]
 [-9.31378031e-17  5.55111512e-16  1.00000000e-01]]

Matrix B (original):
[[0]
 [0]
 [1]]

Matrix B (new):
[[-5.55111512e-17]
 [-1.00000000e-02]
 [ 9.00000000e-01]]

Difference B (new - original):
[[-5.55111512e-17]
 [-1.00000000e-02]
 [-1.00000000e-01]]


Save $\mathbf{A}$ and $\mathbf{B}$ to "model-A-new.tsv" and "model-B-new.tsv" respectively.

In [40]:
# YOUR CHANGES HERE
# Save matrices to tsv files
A_new_df = pd.DataFrame(A_new, columns=['x', 'v', 'a'], index=['x', 'v', 'a'])
B_new_df = pd.DataFrame(B_new, columns=['u'], index=['x', 'v', 'a'])

A_new_df.to_csv('model-A-new.tsv', sep='\t')
B_new_df.to_csv('model-B-new.tsv', sep='\t')

print("Saved new matrices to:")
print("  - model-A-new.tsv")
print("  - model-B-new.tsv")

# Cell 10: Verify saved files
print("\nVerification for model-A-new:")
print(pd.read_csv('model-A-new.tsv', sep='\t', index_col=0))

print("\nVerification for model-B-new")
print(pd.read_csv('model-B-new.tsv', sep='\t', index_col=0))

Saved new matrices to:
  - model-A-new.tsv
  - model-B-new.tsv

Verification for model-A-new:
              x             v             a
x  1.000000e+00  1.100000e+00  9.992007e-16
v  5.820126e-17  9.000000e-01  9.500000e-01
a -9.313780e-17  5.551115e-16  1.100000e+00

Verification for model-B-new
              u
x -5.551115e-17
v -1.000000e-02
a  9.000000e-01


Submit "model-A-new.tsv" and "model-B-new.tsv" in Gradescope.

From my understanding and from the investigation, the quadcopter's poor flight performance stems from a mismatch between the designed control system and the actual hardware dynamics. The prototype was using a suboptimal control matrix with an acceleration feedback gain that was approximately 5% too high (1.950 vs 1.858), which was causing the jerky, over-damped behavior the team observed. This, as we found, wasn't due to a simple calibration error. We instead found that the actual system dynamics differ significantly from the design specifications. The control input produces only 90% of the expected acceleration, acceleration persists 10% longer than designed, and velocity dynamics are 10% more damped than expected. The team had apparently adjusted the control gains in an attempt to compensate for these hardware discrepancies. Their tuning resulted in suboptimal performance as we mentioned. The solution is to recompute the optimal control matrix using the actual system dynamics we've now identified, which will produce a properly tuned controller that accounts for the real hardware behavior rather than the idealized model.

## Part 5: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.
You do not need to provide code for data collection if you did that by manually.

## Part 6: Acknowledgements

If you discussed this assignment with anyone, please acknowledge them here.
If you did this assignment completely on your own, simply write none below.

If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for. If you did not use any other libraries, simply write none below.

If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy. If you did not use any generative AI tools, simply write none below.